In [ ]:
# Import required libraries
import pandas as pd                          # Data manipulation and analysis
import re                                    # Regular expressions for text cleaning
from collections import defaultdict, Counter # Data structures for clustering and frequency counting
from rapidfuzz.distance import Levenshtein   # Levenshtein distance for string similarity

In [ ]:
# Define input and output file paths
# Input: Excel file containing credit transaction and ledger data
# Output: Excel file with categorized and cleaned transaction data
input = "credit_txn_v5.xlsx"
output = "output_file.xlsx"

In [ ]:
# Helper functions for text normalization and string similarity using Levenshtein distance

def normalize(text):
    """
    Normalize text for comparison by:
    - Converting to lowercase
    - Removing all non-alphanumeric characters (keeping only letters, numbers, spaces)
    - Stripping leading/trailing whitespace
    """
    text = str(text).lower()                  # Case-insensitive comparison
    text = re.sub(r'[^a-z0-9 ]', '', text)   # Remove special characters
    return text.strip()

def similarity(a, b):
    """
    Calculate similarity percentage between two strings using Levenshtein distance
    Returns a value between 0 and 100, where 100 means identical
    """
    a, b = normalize(a), normalize(b)
    max_len = max(len(a), len(b))
    if max_len == 0:
        return 100                            # Both empty strings are identical
    # Similarity = (1 - distance/max_length) * 100
    return (1 - Levenshtein.distance(a, b) / max_len) * 100

def blocking_key(text):
    """
    Create a blocking key from first 4 characters of normalized text
    Used for performance optimization - only compare ledgers with same initial characters
    This dramatically reduces comparison pairs in large datasets
    """
    text = normalize(text)
    return text[:4]

In [ ]:
# Levenshtein-based clustering algorithm with frequency-aware blocking
# This function groups similar ledger names into categories using edit distance
# Higher frequency ledgers are processed first to become category representatives

def levenshtein_cluster_by_frequency_fast(ledger_names, threshold=82):
    """
    Cluster ledger names based on Levenshtein similarity with performance optimization
    
    Parameters:
    - ledger_names: list of ledger names to cluster
    - threshold: minimum similarity percentage (0-100) to group ledgers
    
    Returns:
    - category_map: dict mapping category name to list of ledger names in that category
    """
    # Count frequency of each unique ledger name
    freq = Counter(ledger_names)

    # Sort ledgers by frequency (highest first) - frequent names become category representatives
    unique_ledgers = sorted(
        freq.keys(),
        key=lambda x: freq[x],
        reverse=True                                               # Higher frequency first
    )

    # Create blocks by first 4 characters to optimize comparison (blocking strategy)
    # This reduces the number of pairwise comparisons needed
    blocks = defaultdict(list)
    for ledger in unique_ledgers:
        blocks[blocking_key(ledger)].append(ledger)

    # Track visited ledgers and build category mapping
    visited = {}
    category_map = {}

    # Process each block of ledgers with same initial characters
    for block_ledgers in blocks.values():
        n = len(block_ledgers)

        # For each unvisited ledger, create a new category
        for i in range(n):
            ledger_i = block_ledgers[i]

            if visited.get(ledger_i, False):
                continue                                          # Skip already categorized ledgers

            # Start new category with this ledger as the representative
            category_name = ledger_i
            category_map[category_name] = [ledger_i]
            visited[ledger_i] = True

            # Try to match remaining unvisited ledgers to this category
            for j in range(i + 1, n):
                ledger_j = block_ledgers[j]

                if visited.get(ledger_j, False):
                    continue                                      # Skip already categorized ledgers

                # If similarity is above threshold, add to this category
                if similarity(ledger_i, ledger_j) >= threshold:
                    category_map[category_name].append(ledger_j)
                    visited[ledger_j] = True

    return category_map

In [ ]:
# Load transaction data
df = pd.read_excel(input)

# Perform clustering of similar ledger names using Levenshtein distance
# Threshold of 82% similarity means ledgers with 82% matching characters will be grouped
clusters = levenshtein_cluster_by_frequency_fast(
    df["Ledger Name"].tolist(),
    threshold=82                                                   # 82% similarity threshold for clustering
)

In [ ]:
# Build a reverse mapping from individual ledger names to their assigned category
# This allows quick lookup of which category each ledger belongs to

def build_ledger_category_map(category_map):
    """
    Convert category_map (category -> [ledgers]) to a ledger -> category mapping
    
    Parameters:
    - category_map: dict mapping category names to lists of ledger names
    
    Returns:
    - ledger_to_category: dict mapping individual ledger names to their category
    """
    ledger_to_category = {}
    for category, ledgers in category_map.items():
        for ledger in ledgers:
            ledger_to_category[ledger] = category                  # Map each ledger to its category
    return ledger_to_category

In [ ]:
# Convert clustering results to ledger-to-category mapping
ledger_to_category = build_ledger_category_map(clusters)

In [ ]:
# Add the Ledger Category column to the dataframe
df["Ledger Category"] = df["Ledger Name"].map(ledger_to_category)

# Safety fallback: If a ledger wasn't categorized, use the original ledger name
# This should be rare but ensures no null values in the category column
df["Ledger Category"] = df["Ledger Category"].fillna(df["Ledger Name"])

In [ ]:
# Mark low-frequency ledgers as "OTHER" category
# Ledgers with fewer than 5 transactions are considered noise/outliers
# Consolidating them improves data quality and analysis clarity

# 1️⃣ Calculate frequency of each ledger name in the dataset
ledger_freq = df['Ledger Name'].value_counts()

# 2️⃣ Identify ledgers with frequency less than minimum threshold
low_freq_ledgers = ledger_freq[ledger_freq < 5].index

# 3️⃣ Update the Ledger Category to "OTHER" for low-frequency ledgers
df.loc[
    df['Ledger Name'].isin(low_freq_ledgers),
    'Ledger Category'
] = "OTHER"

In [ ]:
# Clean illegal Excel control characters and export final results
# Excel does not support control characters (0x00-0x1F) in cell values
# Removing these ensures the output file is compatible with all Excel versions

import re

# Regular expression to match all control characters
ILLEGAL_EXCEL_CHARS = re.compile(r'[\x00-\x1F]')

def clean_excel_text(val):
    """Remove illegal control characters from text values"""
    if isinstance(val, str):
        return ILLEGAL_EXCEL_CHARS.sub('', val)
    return val

# Apply cleaning to all cells in the dataframe
df = df.applymap(clean_excel_text)

# Export the processed and categorized data to Excel
# Final output includes:
# - Original transaction data
# - New "Ledger Category" column with normalized/clustered ledger names
# - All text cleaned of control characters
df.to_excel(
    output,
    index=False                                                    # Don't include row indices
)

C:\Users\subha\AppData\Local\Temp\ipykernel_1048\2910059861.py:10: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df = df.applymap(clean_excel_text)
